In [1]:
import torch
from torch import nn

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


In [2]:
class FashionMNISTModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [3]:
model = FashionMNISTModel()
print(model)

FashionMNISTModel(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)


In [4]:
# Create a fake batch of 1 image (28x28)
x = torch.randn(1, 28, 28)

# Pass it through the model
logits = model(x)

print("Output shape:", logits.shape)
print("Logits:", logits)

Output shape: torch.Size([1, 10])
Logits: tensor([[-0.1294,  0.0078, -0.1516, -0.0585, -0.0339,  0.0490,  0.0601, -0.0381,
         -0.0990,  0.1301]], grad_fn=<AddmmBackward0>)


In [5]:
# Convert logits to predicted class
predicted_class = logits.argmax(dim=1)

print("Predicted class:", predicted_class.item())

Predicted class: 9


In [6]:
# Fake true label for our fake image
y = torch.tensor([3])

# Loss function for multi-class classification
loss_fn = nn.CrossEntropyLoss()

# Calculate loss
loss = loss_fn(logits, y)

print("Loss:", loss.item())

Loss: 2.338320732116699


In [7]:
# Define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Zero gradients
optimizer.zero_grad()

# Backward pass (compute gradients)
loss.backward()

# Update weights
optimizer.step()

print("Updated weights!")

Updated weights!


In [8]:
# Run forward pass again
new_logits = model(x)

# Calculate new loss
new_loss = loss_fn(new_logits, y)

print("Old loss:", loss.item())
print("New loss:", new_loss.item())

Old loss: 2.338320732116699
New loss: 2.1971659660339355


In [9]:
from torchvision import datasets
from torchvision.transforms import ToTensor

# Download training data
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

# Download test data
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

print("Training samples:", len(training_data))
print("Test samples:", len(test_data))

100%|██████████| 26.4M/26.4M [00:01<00:00, 18.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 313kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.63MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 22.3MB/s]

Training samples: 60000
Test samples: 10000


In [10]:
print(type(training_data))

<class 'torchvision.datasets.mnist.FashionMNIST'>


In [11]:
from torch.utils.data import DataLoader

batch_size = 64

train_dataloader = DataLoader(
    training_data,
    batch_size=batch_size,
    shuffle=True
)

test_dataloader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False
)

print("Train batches:", len(train_dataloader))
print("Test batches:", len(test_dataloader))

Train batches: 938
Test batches: 157


In [12]:
for X, y in train_dataloader:
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    break

X shape: torch.Size([64, 1, 28, 28])
y shape: torch.Size([64])


In [13]:
# Loss function and optimizer (reuse or redefine)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    for batch, (X, y) in enumerate(dataloader):

        # Forward pass
        pred = model(X)

        # Compute loss
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Print progress occasionally
        if batch % 100 == 0:
            loss_value = loss.item()
            current = batch * len(X)
            print(f"loss: {loss_value:>7f}  [{current:>5d}/{size:>5d}]")

In [14]:
print("Starting training...")
train_loop(train_dataloader, model, loss_fn, optimizer)
print("Done!")

Starting training...
loss: 2.296766  [    0/60000]
loss: 2.222379  [ 6400/60000]
loss: 2.054512  [12800/60000]
loss: 1.753876  [19200/60000]
loss: 1.444568  [25600/60000]
loss: 1.260023  [32000/60000]
loss: 1.027618  [38400/60000]
loss: 0.923829  [44800/60000]
loss: 0.952937  [51200/60000]
loss: 0.940249  [57600/60000]
Done!


In [15]:
def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size

    print(f"Test Error: \n Accuracy: {100*correct:.1f}%, Avg loss: {test_loss:.6f}\n")

In [16]:
test_loop(test_dataloader, model, loss_fn)

Test Error: 
 Accuracy: 66.7%, Avg loss: 0.852434



In [17]:
epochs = 5

for epoch in range(epochs):
    print(f"Epoch {epoch+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)

print("Training complete!")

Epoch 1
-------------------------------
loss: 0.873734  [    0/60000]
loss: 0.747182  [ 6400/60000]
loss: 0.765404  [12800/60000]
loss: 0.878829  [19200/60000]
loss: 0.829055  [25600/60000]
loss: 0.749056  [32000/60000]
loss: 0.708439  [38400/60000]
loss: 0.817384  [44800/60000]
loss: 0.603564  [51200/60000]
loss: 0.550394  [57600/60000]
Test Error: 
 Accuracy: 73.9%, Avg loss: 0.704273

Epoch 2
-------------------------------
loss: 0.525942  [    0/60000]
loss: 0.673202  [ 6400/60000]
loss: 0.580833  [12800/60000]
loss: 0.729750  [19200/60000]
loss: 0.632340  [25600/60000]
loss: 0.545237  [32000/60000]
loss: 0.775047  [38400/60000]
loss: 0.475871  [44800/60000]
loss: 0.491336  [51200/60000]
loss: 0.483767  [57600/60000]
Test Error: 
 Accuracy: 79.1%, Avg loss: 0.594298

Epoch 3
-------------------------------
loss: 0.633814  [    0/60000]
loss: 0.464785  [ 6400/60000]
loss: 0.731562  [12800/60000]
loss: 0.505826  [19200/60000]
loss: 0.370570  [25600/60000]
loss: 0.480236  [32000/60000

In [18]:
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    total_loss = 0

    for batch, (X, y) in enumerate(dataloader):

        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Train Loss: {avg_loss:.4f}")

In [23]:
epochs = 10

for epoch in range(epochs):
    print(f"Epoch {epoch+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)

print("Training complete!")

Epoch 1
-------------------------------
Train Loss: 0.5148
Test Error: 
 Accuracy: 86.2%, Avg loss: 0.386709

Epoch 2
-------------------------------
Train Loss: 0.3330
Test Error: 
 Accuracy: 87.0%, Avg loss: 0.344394

Epoch 3
-------------------------------
Train Loss: 0.2862
Test Error: 
 Accuracy: 89.2%, Avg loss: 0.292777

Epoch 4
-------------------------------
Train Loss: 0.2569
Test Error: 
 Accuracy: 90.1%, Avg loss: 0.274914

Epoch 5
-------------------------------
Train Loss: 0.2349
Test Error: 
 Accuracy: 90.2%, Avg loss: 0.265799

Epoch 6
-------------------------------
Train Loss: 0.2138
Test Error: 
 Accuracy: 90.9%, Avg loss: 0.258921

Epoch 7
-------------------------------
Train Loss: 0.1977
Test Error: 
 Accuracy: 91.2%, Avg loss: 0.247731

Epoch 8
-------------------------------
Train Loss: 0.1816
Test Error: 
 Accuracy: 91.2%, Avg loss: 0.242422

Epoch 9
-------------------------------
Train Loss: 0.1687
Test Error: 
 Accuracy: 91.5%, Avg loss: 0.239886

Epoch 10
-

In [20]:

import torch
import torch.nn as nn

In [21]:
class FashionMNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_stack = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_stack(x)
        logits = self.classifier(x)
        return logits

In [22]:
model = FashionMNIST_CNN()
print(model)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

FashionMNIST_CNN(
  (conv_stack): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1568, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)
